# Formula One World Championship Data Cleaning & Preprocessing

## Objective
This notebook prepares historical Formula One race data (1950–2020) for downstream analytics and dashboarding. The workflow focuses on cleaning, integrating, and feature engineering while preserving data integrity and reproducibility.

**Expected outcome:** a consistent, analysis-ready dataset exported as `cleaned_f1_dataset.csv`.


## Import Libraries
This section imports the core Python libraries required for data manipulation and numerical operations.

**Why this is necessary:** standardized libraries ensure reliable and reproducible preprocessing steps.

**Expected outcome:** all dependencies are loaded before data operations begin.


In [18]:
import pandas as pd
import numpy as np


## Load Dataset
Here we load the required relational tables from the Kaggle Formula 1 dataset: drivers, constructors, results, races, circuits, and status.

**Why this is necessary:** these core tables contain complementary race, participant, and outcome information needed for integration.

**Expected outcome:** raw datasets are available as DataFrames for exploration and cleaning.


In [19]:
# Load datasets

drivers = pd.read_csv("data/drivers.csv")
constructors = pd.read_csv("data/constructors.csv")
results = pd.read_csv("data/results.csv")
races = pd.read_csv("data/races.csv")
circuits = pd.read_csv("data/circuits.csv")
status = pd.read_csv("data/status.csv")

# Store all DataFrames for standardized cleaning operations
dfs = {
    "drivers": drivers,
    "constructors": constructors,
    "results": results,
    "races": races,
    "circuits": circuits,
    "status": status,
}


## Data Exploration
We perform an initial inspection of the results table to understand structure, data types, missing values, and duplicate records.

**Why this is necessary:** early profiling helps identify cleaning priorities and validates assumptions before transformation.

**Expected outcome:** a clear baseline view of data quality issues to address in the cleaning phase.


In [20]:
# Inspect dataset structure and basic quality indicators
results.head()


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [21]:
results.info()


<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  str    
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  str    
 7   positionText     26759 non-null  str    
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  str    
 12  milliseconds     26759 non-null  str    
 13  fastestLap       26759 non-null  str    
 14  rank             26759 non-null  str    
 15  fastestLapTime   26759 non-null  str    
 16  fastestLapSpeed  26759 non-null  str    
 17  statusId         26759 

In [22]:
results.isnull().sum()


resultId           0
raceId             0
driverId           0
constructorId      0
number             0
grid               0
position           0
positionText       0
positionOrder      0
points             0
laps               0
time               0
milliseconds       0
fastestLap         0
rank               0
fastestLapTime     0
fastestLapSpeed    0
statusId           0
dtype: int64

In [23]:
results.duplicated().sum()


np.int64(0)

The exploration confirms the schema and highlights where null handling and type conversions are required before transformation.


## Data Cleaning
This section standardizes missing values, removes duplicate rows, and converts relevant columns to suitable data types.

**Why this is necessary:** clean and typed data prevents merge issues and improves analytical consistency.

**Expected outcome:** all source tables are deduplicated and key temporal/numeric fields are properly formatted.


In [24]:
# Handle missing values represented as \N
for _, df in dfs.items():
    df.replace(r"\N", np.nan, inplace=True)

# Remove duplicate records in each source table
for _, df in dfs.items():
    df.drop_duplicates(inplace=True)

# Convert race date to datetime
races["date"] = pd.to_datetime(races["date"])

# Convert relevant result columns to numeric types
numeric_columns = [
    "grid",
    "position",
    "positionOrder",
    "points",
    "laps",
    "milliseconds",
    "fastestLapSpeed",
]

for col in numeric_columns:
    if col in results.columns:
        results[col] = pd.to_numeric(results[col], errors="coerce")


Data cleaning is complete, and all tables are now ready for relational integration.


## Data Integration
We merge the cleaned tables into a single analytical dataset using foreign keys across results, drivers, constructors, races, circuits, and status.

**Why this is necessary:** combining normalized tables creates a comprehensive race-level view for analysis.

**Expected outcome:** one merged DataFrame with clear, human-readable column names and no ambiguous suffixes.


In [25]:
# Merge results with drivers and clarify overlapping number fields
merged = results.merge(drivers, on="driverId", how="left").rename(
    columns={"number_x": "Car Number", "number_y": "Driver Number"}
)

# Merge constructors and immediately rename overlapping fields
merged = merged.merge(
    constructors,
    on="constructorId",
    how="left",
    suffixes=("", "_constructor"),
).rename(
    columns={
        "name": "Constructor",
        "nationality_constructor": "Constructor Nationality",
    }
)

# Merge races and assign race name
merged = merged.merge(
    races,
    on="raceId",
    how="left",
    suffixes=("", "_race"),
).rename(columns={"name": "Race"})

# Merge circuits and assign circuit name
merged = merged.merge(
    circuits,
    on="circuitId",
    how="left",
    suffixes=("", "_circuit"),
).rename(columns={"name": "Circuit"})

# Merge status table for race completion details
merged = merged.merge(status, on="statusId", how="left")


The integrated dataset now combines race context, driver identity, constructor information, and final status in one consistent table.


## Feature Engineering
In this section, we derive analysis-friendly features such as full driver name, win/podium indicators, completion flag, decade grouping, and categorical buckets for position and points.

**Why this is necessary:** engineered features improve readability and support downstream descriptive analysis and visualization.

**Expected outcome:** the merged dataset includes additional, interpretable analytical columns.


In [26]:
# Create new analytical features
merged["Driver"] = merged["forename"] + " " + merged["surname"]

merged["Winner"] = np.where(merged["positionOrder"] == 1, "Yes", "No")
merged["Podium"] = np.where(merged["positionOrder"] <= 3, "Yes", "No")
merged["Finished"] = np.where(merged["status"] == "Finished", "Yes", "No")
merged["Decade"] = (merged["year"] // 10) * 10

merged["Position Category"] = pd.cut(
    merged["positionOrder"],
    bins=[0, 1, 3, 10, 100],
    labels=["Winner", "Podium", "Top 10", "Others"],
)

merged["Points Category"] = pd.cut(
    merged["points"],
    bins=[-1, 0, 10, 25, 100],
    labels=["0", "1-10", "11-25", "25+"],
)


## Final Dataset
We now select the final columns required for reporting and model-ready analytics.

**Why this is necessary:** column selection keeps the output focused, clean, and aligned to project objectives.

**Expected outcome:** a compact final DataFrame with race, driver, constructor, performance, and engineered attributes.


In [27]:
# Select final curated columns
final_df = merged[
    [
        "raceId",
        "year",
        "round",
        "date",
        "Race",
        "Circuit",
        "location",
        "country",
        "Driver",
        "nationality",
        "Constructor",
        "grid",
        "positionOrder",
        "points",
        "laps",
        "fastestLapSpeed",
        "Winner",
        "Podium",
        "Finished",
        "Position Category",
        "Points Category",
        "status",
        "Decade",
    ]
]


In [28]:
final_df.head()


,raceId,year,round,date,Race,Circuit,location,country,Driver,nationality,...,points,laps,fastestLapSpeed,Winner,Podium,Finished,Position Category,Points Category,status,Decade
0,18,2008,1,2008-03-16,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,Lewis Hamilton,British,...,10.0,58,218.300,Yes,Yes,Yes,Winner,1-10,Finished,2000
1,18,2008,1,2008-03-16,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,Nick Heidfeld,German,...,8.0,58,217.586,No,Yes,Yes,Podium,1-10,Finished,2000
2,18,2008,1,2008-03-16,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,Nico Rosberg,German,...,6.0,58,216.719,No,Yes,Yes,Podium,1-10,Finished,2000
3,18,2008,1,2008-03-16,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,Fernando Alonso,Spanish,...,5.0,58,215.464,No,No,Yes,Top 10,1-10,Finished,2000
4,18,2008,1,2008-03-16,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,Heikki Kovalainen,Finnish,...,4.0,58,218.385,No,No,Yes,Top 10,1-10,Finished,2000


In [29]:
final_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   raceId             26759 non-null  int64         
 1   year               26759 non-null  int64         
 2   round              26759 non-null  int64         
 3   date               26759 non-null  datetime64[us]
 4   Race               26759 non-null  str           
 5   Circuit            26759 non-null  str           
 6   location           26759 non-null  str           
 7   country            26759 non-null  str           
 8   Driver             26759 non-null  str           
 9   nationality        26759 non-null  str           
 10  Constructor        26759 non-null  str           
 11  grid               26759 non-null  int64         
 12  positionOrder      26759 non-null  int64         
 13  points             26759 non-null  float64       
 14  laps             

In [30]:
final_df.isnull().sum()


raceId                   0
year                     0
round                    0
date                     0
Race                     0
Circuit                  0
location                 0
country                  0
Driver                   0
nationality              0
Constructor              0
grid                     0
positionOrder            0
points                   0
laps                     0
fastestLapSpeed      18507
Winner                   0
Podium                   0
Finished                 0
Position Category        0
Points Category          0
status                   0
Decade                   0
dtype: int64

In [31]:
print(final_df.shape)


(26759, 23)


The final dataset is structurally consistent, retains expected dimensions, and is ready for downstream analytics and visualization.


## Export Dataset
This step exports the cleaned and integrated dataset for external analysis and reporting.

**Why this is necessary:** exporting creates a portable artifact for tools such as Google Looker Studio.

**Expected outcome:** `cleaned_f1_dataset.csv` is generated successfully in the project directory.


In [32]:
# Export cleaned dataset
final_df.to_csv("cleaned_f1_dataset.csv", index=False)


## Conclusion
This notebook cleaned, standardized, and integrated Formula One historical race data into a professional, analysis-ready dataset. The final output preserves the original workflow while improving clarity, reproducibility, and submission quality for data analytics review.
